# Algorithmic Methods of Data Mining and Laboratory

# Lab Session 03 - Information Retrieval

Information Retrieval is a field of data science that uses search engines to find materials from a large (unstructured) collection in order to satisfy a user's information need.

In [ ]:
import pandas as pd
import sklearn as sk
import numpy as np
import re
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.datasets import fetch_20newsgroups
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Fetch and investigate the dataset.
dataset = fetch_20newsgroups(subset='train')
print (dataset.data[0])

# Explore the contents of the dataset.
print(len(dataset))
dataset.keys() # This method returns the column names.

# Construct the corpus as a list of strings, with each string representing a full document.
corpus = dataset["data"]

## Text Mining

Generally speaking, the extracted raw text undergoes a pre-processing pipeline that consists of the following steps:
1. **Tokenization:** A sequence of words or tokens is extracted from the document.
2. **Stopword Removal:** Terms that carry little semantic value are removed to reduce noise.
3. **Normalization:** Text features are standardized to avoid mismatches related to syntax or character case.
4. **Stemming:** Words get truncated to their root in order to group similar concepts together.

In [ ]:
def clean(doc):
  # Clean the document by removing irrelevant elements and solving case mismatches.
  doc = re.sub(r"[^a-zA-Z0-9\s]", "", doc) # Keep only letters and numbers.
  doc = re.sub(r"\n", " ", doc) # Remove newline.
  doc = doc.lower() # Convert every word to lowercase.
  doc = doc.strip() # Remove leading and trailing whitespaces.
  doc = re.sub(' +', ' ', doc) # Remove multiple whitespaces.
  return doc

corpus_cleaned = []
for doc in corpus:
  corpus_cleaned.append(clean(doc))
corpus_cleaned[0]

In [ ]:
stemmer = SnowballStemmer("english")

def tokenize_and_stem(doc):
  # Perform stemming on a document.
  words = doc.split(" ")
  stemmed_words = [stemmer.stem(word) for word in words]
  return stemmed_words

corpus_stemmed = []
for doc in tqdm(corpus_cleaned):
  corpus_stemmed.append(tokenize_and_stem(doc))
corpus_stemmed[0]

Note that the same pre-processing pipeline must be applied to query vectors as well.

## Document Encoding

Generally speaking, a document can be represented as a text vector, which simply reports the raw frequencies of each term in the vocabulary, or using the TF-IDF representation, which considers term relevance as well.

### Text Vector

In [ ]:
# Create a new corpus and represent it as a text vector.
new_corpus = [" ".join(words) for words in corpus_stemmed]
count_vectorizer = CountVectorizer(stop_words='english', min_df=0.005)
encoded_corpus = count_vectorizer.fit_transform(new_corpus) # A matrix in which each row corresponds to a document and each column denotes a term in the vocabulary.
encoded_corpus # Only 805959 elements are stored in the vector, making it sparse.

In [ ]:
# Look at the terms in the vocabulary.
vocab = count_vectorizer.get_feature_names_out()
print(vocab)

# Represent the text vector as a DataFrame.
text_df = pd.DataFrame(encoded_corpus.toarray(), columns=count_vectorizer.get_feature_names_out())
text_df

# Find the number of documents in the corpus that contain each term in the vocabulary.
text_df_sum = text_df.sum(axis=0)
text_df_sum

In [ ]:
# Get the 20 most frequent terms in the corpus.
df_sum_sorted = df_sum.sort_values(ascending=False)
first_20_terms = df_sum_sorted[:20]

# Create a bar plot showing the frequency of the first 20 terms.
plt.figure(figsize=(10, 6))
plt.bar(first_20_terms.index, first_20_terms.values)
plt.xlabel("Terms")
plt.ylabel("Frequency")
plt.title("Frequency of the First 20 Terms")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Determine the documents that are compatible with a given query.
query = "sports car"
processed_query = " ".join(tokenize_and_stem(clean(query))) # Pre-process the query.
encoded_query = count_vectorizer.transform([processed_query]) # Encode the query as a text vector.

# Find the documents that contain at least one term of the query.
non_zero_indices = encoded_query.nonzero()[1]
subset_df = text_df.iloc[:,non_zero_indices]
vocab[non_zero_indices]
subset_df

# Find the documents that contain all terms of the query.
subset_df = subset_df.loc[(subset_df>0).all(1)]
subset_df

# Sort the compatible documents and look at the contents of the first three documents.
subset_df_sorted = subset_df.sum(axis=1).sort_values(ascending=False)
subset_df = subset_df.loc[subset_df_sorted.index]
subset_df

cont = 0
for i in subset_df.index:
  print("-" * 20, "DOC: ", i, "-" * 20)
  print(corpus[i])
  print("\n"*5)
  cont += 1
  if cont > 2:
    break

### TF-IDF Representation

In [ ]:
# Create a new corpus and represent it as a TF-IDF vector.
tfidf_vectorizer = TfidfVectorizer(stop_words='english', use_idf=True, min_df=0.005)
encoded_corpus = tfidf_vectorizer.fit_transform(new_corpus)
df_tfidf = pd.DataFrame(encoded_corpus.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
df_tfidf

In [ ]:
# Get the 20 most frequent terms in the corpus.
df_sum = df_tfidf.sum(axis=0)
df_sum_sorted = df_sum.sort_values(ascending=False)
first_20_terms = df_sum_sorted[:20]

# Create a bar plot showing the frequency of the first 20 terms.
plt.figure(figsize=(10, 6))
plt.bar(first_20_terms.index, first_20_terms.values)
plt.xlabel("Terms")
plt.ylabel("TF-IDF")
plt.title("TF-IDF of the First 20 Terms")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Determine the documents that are compatible with a given query.
query = "sports car"
processed_query = " ".join(tokenize_and_stem(clean(query))) # Pre-process the query.
encoded_query = tfidf_vectorizer.transform([processed_query]) # Encode the query as a text vector.

# Find the closest documents by computing the cosine similarity between the encoded query and the TF-IDF representations.
cossims = cosine_similarity(df_tfidf, encoded_query)
cossims

# Look at the contents of the four closest documents.
for query_id in range(1):
  cont = 0
  for doc_id in cossims[:,query_id].argsort()[::-1]:
    print("-" * 20, "DOC: ", doc_id, "-" * 20)
    print(corpus[doc_id])
    print("\n"*5)
    cont += 1
    if cont > 3:
      break

## Inverted Indexing

A search engine can be made more efficient by introducing an "inverted index" data structure, which, for every term in a vocabulary, stores a posting list containing all the documents that contain that term.

In [ ]:
# Implement an inverted index data structure using a term:document_id dictionary.
inverted_index = {}
for doc_id, words in enumerate(corpus_stemmed):
  for word in words:
    if word in inverted_index:
      inverted_index[word].add(doc_id)
    else:
      inverted_index[word] = set([doc_id])

# Look at some terms in the dictionary.
cont = 0
for key, docs in inverted_index.items():
  print(key, ":", docs)
  cont += 1
  if cont > 4:
    break